# Download the Bulk 10-K Filing Corpus

| Item | Description |
|---|---|
| Goal | Download sec-api.io's complete Form 10-K dataset: 394 monthly ZIP containers holding 305,140 annual reports, 34 GB, from October 1993 to the present. Every filing arrives with a `metadata.json` carrying the CIK, the exact form type, the period, and the SEC.gov URL, so the containers are the whole enumeration in 394 requests, and the local archive the extraction can be checked against. |
| Input | The sec-api.io dataset manifest, `/datasets/form-10k-content.json`, and the containers, `/datasets/form-10k-content/<year>/<year>-<month>.zip`. The key `SEC_API_KEY` in `.env` in the D folder. sec-api.io is a paid service. |
| Output | In the D3 Raw folder: `form10k_index.json`, the manifest with every container's size and record count; `form10k/<year>/<year>-<month>.zip`, the containers. |
| Run | Last run 2026-08-20 as a SLURM job; the cell outputs shown are from that run. Not rerun: 34 GB from a paid API, and the containers are on disk. |

## Step 1. Import packages

In [ ]:
import json
import os
import time
import zipfile
from pathlib import Path

import pandas as pd
import requests

## Step 2. Define file locations

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("D - Data Collection and Processing/D3 - SEC Form 10-K Corporate Filings")
NOTEBOOK_NAME = "D3.1 - SEC 10-K - Download Bulk Containers"
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
CONTAINER_FOLDER = RAW_FOLDER / "form10k"  # one ZIP per month
ENV_FILE = PROJECT_FOLDER / "D - Data Collection and Processing" / ".env"  # holds SEC_API_KEY

CONTAINER_FOLDER.mkdir(parents=True, exist_ok=True)

## Step 3. Read the API key and set the pace

Each request transfers tens of megabytes, so bandwidth is the limit, not the request count. Two requests per second is cheap insurance against the key being blocked for the rest of a second.

In [1]:
SECONDS_BETWEEN_REQUESTS = 0.5
MAX_ATTEMPTS = 5
TIMEOUT_SECONDS = 1800  # one container can take many minutes
TRANSIENT_STATUS = {408, 425, 500, 502, 503, 504}

API_KEY = os.environ.get("SEC_API_KEY")
if not API_KEY and ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():  # one KEY=value per line
        name, _, value = line.strip().partition("=")
        if name == "SEC_API_KEY" and value:
            API_KEY = value.strip().strip('"').strip("'")
assert API_KEY, f"SEC_API_KEY is not set and {ENV_FILE} does not supply it"

print(f"containers -> {CONTAINER_FOLDER}")

containers -> /scratch2/acroh/projects/llm_method/data/raw/sec/form10k


## Step 4. Define one API request and one container download

A request carries the key in a header and retries a transient failure. On a rate-limit answer it pauses a quarter second without spending a retry. A container is written under a temporary name, its byte count is compared with the manifest, and only then is it renamed into place. A short download is the failure a bulk transfer actually has, and it is silent: the ZIP opens and simply holds fewer filings.

In [ ]:
BASE_URL = "https://api.sec-api.io"
RATE_LIMIT_PAUSE = 0.25  # a 429 blocks the key only for the rest of the second, so a short pause is the right answer
MAX_RATE_LIMIT_WAITS = 60

session = requests.Session()
request_log = {"requests": 0, "retries": 0, "rate_limited": 0, "last_request": 0.0}


def secapi_get(url, params=None, stream=False):
    """One GET. Retries a transient failure, pauses on a rate limit, raises on anything else."""
    last_error = "no attempt made"
    attempt = 0
    waits = 0
    while attempt < MAX_ATTEMPTS:
        wait = SECONDS_BETWEEN_REQUESTS - (time.monotonic() - request_log["last_request"])
        if wait > 0:
            time.sleep(wait)
        request_log["last_request"] = time.monotonic()
        request_log["requests"] += 1
        try:
            response = session.get(url, params=params or {}, headers={"Authorization": API_KEY}, timeout=TIMEOUT_SECONDS, stream=stream)
        except requests.RequestException as exc:
            last_error = f"{type(exc).__name__}: {exc}"
        else:
            if response.status_code == 200:
                return response
            if response.status_code == 429:
                waits += 1
                request_log["rate_limited"] += 1
                assert waits <= MAX_RATE_LIMIT_WAITS, f"still rate limited after {waits} pauses for {url}"
                time.sleep(RATE_LIMIT_PAUSE)
                continue
            if response.status_code not in TRANSIENT_STATUS:
                raise RuntimeError(f"HTTP {response.status_code} for {url}: {response.text[:200]}")
            last_error = f"HTTP {response.status_code}"
        request_log["retries"] += 1
        attempt += 1
        time.sleep(2.0 ** attempt)
    raise RuntimeError(f"{MAX_ATTEMPTS} attempts failed for {url}: {last_error}")


def download_container(key, target, expected_size):
    """Download one monthly container, refuse to keep it unless its size matches the manifest, and return the bytes written."""
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + ".partial")
    with secapi_get(f"{BASE_URL}/datasets/form-10k-content/{key}", {"token": API_KEY}, stream=True) as response, partial.open("wb") as out:
        written = 0
        for chunk in response.iter_content(chunk_size=1 << 20):
            if chunk:
                out.write(chunk)
                written += len(chunk)
    if written != expected_size:
        partial.unlink(missing_ok=True)
        raise RuntimeError(f"{key}: expected {expected_size:,} bytes, got {written:,}; refusing to keep a short container")
    partial.replace(target)
    return written

## Step 5. Fetch and save the manifest

The manifest states each container's exact size and record count. Every later completeness check is against this file, and without it the archive could only be checked against itself.

In [2]:
index = secapi_get(f"{BASE_URL}/datasets/form-10k-content.json", {"token": API_KEY}).json()
(RAW_FOLDER / "form10k_index.json").write_text(json.dumps(index, indent=1))

containers = sorted(index["containers"], key=lambda c: c["key"])
manifest = pd.DataFrame(containers)
manifest["year"] = manifest["key"].str.split("/").str[0].astype(int)

print(f"{index['name']}")
print(f"  {index['totalRecords']:,} records, {index['totalSize'] / 1e9:.1f} GB, {len(containers)} containers")
print(f"  earliest {index['earliestSampleDate'][:10]}, updated {index['updatedAt'][:10]}")
print(f"  form types: {index['formTypes']}")
print()
by_year = manifest.groupby("year").agg(containers=("key", "size"), records=("records", "sum"), bytes=("size", "sum"))
by_year["gb"] = by_year.pop("bytes") / 1e9
print(by_year.to_string())

Form 10-K - Annual Reports - Filing Contents
  305,140 records, 34.0 GB, 394 containers
  earliest 1993-10-01, updated 2026-08-20
  form types: ['10-K', '10-K/A', '10-K405', '10-K405/A', '10-KSB', '10-KSB/A', '10-KT', '10-KT/A']

      containers  records        gb
year                               
1993           2        5  0.000264
1994          12     2426  0.195069
1995          12     4263  0.302476
1996          12     7931  0.509616
1997          12    12193  0.828302
1998          12    12262  0.883142
1999          12    12017  0.877863
2000          12    11383  0.838292
2001          12    10798  0.537126
2002          12    10644  0.599107
2003          12    10123  0.700694
2004          12    10193  0.728072
2005          12    10725  0.827067
2006          12     9944  0.875531
2007          12     9643  0.968208
2008          12    10176  0.983235
2009          12    11785  1.180356
2010          12    11029  1.162101
2011          12    10579  1.143399
2012          

## Step 6. Download every container

A container already on disk at its manifest size is skipped, so the notebook picks up where it stopped.

In [3]:
done = skipped = 0
failures = []

for i, container in enumerate(containers, 1):
    target = CONTAINER_FOLDER / container["key"]
    if target.exists() and target.stat().st_size == container["size"]:
        skipped += 1
        continue
    try:
        download_container(container["key"], target, expected_size=container["size"])
        done += 1
    except RuntimeError as exc:
        failures.append({"key": container["key"], "error": str(exc)[:200]})
        continue
    if done % 25 == 0:
        print(f"  {i}/{len(containers)}  downloaded={done} skipped={skipped} failed={len(failures)}")

print(f"\ndownloaded {done}, already present {skipped}, failed {len(failures)}")
print(f"{request_log['requests']} requests, {request_log['retries']} retries")
if failures:
    print(pd.DataFrame(failures).to_string(index=False))


downloaded 0, already present 394, failed 0
1 requests, 0 retries


## Step 7. Check the archive against the manifest

Four checks. The first three stop the notebook if they fail.

- Every container is on disk.
- Every container is exactly its manifest size, which catches the silent short read.
- Every container holds one document and one metadata record per filing.
- The manifest's `records` field reconciles container by container. For the newest containers it counts ZIP entries, two per filing, rather than filings, so a total compared with a total would read 4,350 filings as missing when none is.

In [4]:
on_disk = {p.relative_to(CONTAINER_FOLDER).as_posix(): p for p in CONTAINER_FOLDER.rglob("*.zip")}
expected = {c["key"]: c for c in containers}

missing = sorted(set(expected) - set(on_disk))
assert not missing, f"{len(missing)} containers missing, e.g. {missing[:5]}"
print(f"1) all {len(expected)} containers on disk")

wrong = [k for k, c in expected.items() if on_disk[k].stat().st_size != c["size"]]
assert not wrong, f"{len(wrong)} containers are not their manifest size, e.g. {wrong[:5]}"
print(f"2) all {len(expected)} containers match their manifest byte size")

per_container = {}
for key, path in on_disk.items():
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
    per_container[key] = {
        "filings": sum(1 for n in names if n.endswith("metadata.json")),
        "documents": sum(1 for n in names if not n.endswith(".json")),
        "entries": len(names),
    }

lopsided = {k: v for k, v in per_container.items() if v["filings"] != v["documents"]}
assert not lopsided, f"{len(lopsided)} containers do not pair each filing with one document: {list(lopsided)[:5]}"
counted = sum(v["filings"] for v in per_container.values())
print(f"3) every filing in all {len(per_container)} containers has exactly one document and one metadata record ({counted:,} filings)")

as_filings = [k for k, c in expected.items() if per_container[k]["filings"] == c["records"]]
as_entries = [k for k, c in expected.items() if per_container[k]["filings"] != c["records"] and per_container[k]["entries"] >= c["records"]]
unexplained = sorted(set(expected) - set(as_filings) - set(as_entries))

print(f"4) manifest `records` reconciles as filings in {len(as_filings)} containers, as ZIP entries in {len(as_entries)}")
for k in as_entries:
    c, v = expected[k], per_container[k]
    print(f"     {k}: {v['filings']:,} filings, {v['entries']:,} entries, manifest says {c['records']:,} (+{v['entries'] - c['records']} on disk)")
assert not unexplained, f"{len(unexplained)} containers match the manifest neither as filings nor as entries, e.g. {unexplained[:5]}"
print(f"   {index['totalRecords'] - counted:,} of the manifest total is that unit change, not missing filings")

1) all 394 containers on disk
2) all 394 containers match their manifest byte size


3) every filing in all 394 containers has exactly one document and one metadata record (300,790 filings)
4) manifest `records` reconciles as filings in 388 containers, as ZIP entries in 6
     2026/2026-03.zip: 3,244 filings, 6,488 entries, manifest says 6,468 (+20 on disk)
     2026/2026-04.zip: 599 filings, 1,198 entries, manifest says 1,196 (+2 on disk)
     2026/2026-05.zip: 186 filings, 372 entries, manifest says 360 (+12 on disk)
     2026/2026-06.zip: 175 filings, 350 entries, manifest says 345 (+5 on disk)
     2026/2026-07.zip: 111 filings, 222 entries, manifest says 219 (+3 on disk)
     2026/2026-08.zip: 89 filings, 178 entries, manifest says 166 (+12 on disk)
   4,350 of the manifest total is that unit change, not missing filings
